# Create a DuckDB table from the NYC 311 API

Run these cells in order with the repository's `.venv` kernel, from the repository root or notebook folder. The API step reuses `nyc_311_api_connection.ipynb`, including its endpoint, selected fields, date filter, sample limit and optional `SOCRATA_APP_TOKEN`.

The SQL step replaces the local exploratory table `"311_service_requests"` in `data/nyc_open_data.db`. It loads only the configured bounded sample (1,000 rows by default); it is not a complete cohort or a historical Vault load. The SQL file requires the registered API DataFrame and cannot be run alone in a separate SQL connection. Detailed source locations are persisted locally; do not commit the database or notebook outputs.

In [11]:
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "docs/PROJECT_PLAN.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from within the project repository.")
DATABASE_PATH = PROJECT_ROOT / "data" / "nyc_open_data.db"
SQL_PATH = PROJECT_ROOT / "sql_queries/create_311_service_request.sql"
API_NOTEBOOK = PROJECT_ROOT / "output/jupyter-notebook/nyc_311_api_connection.ipynb"


## Fetch the configured API sample

This cell sends the HTTPS GET request by running the existing API notebook. Edit the query configuration there before running this cell. A failed request stops execution before table creation; `df` is cleared first to prevent reusing an earlier result.

In [12]:
df = None
capture_metadata = None
get_ipython().run_line_magic("run", f'"{API_NOTEBOOK}"')
if not isinstance(df, pd.DataFrame) or capture_metadata is None:
    raise RuntimeError("The API notebook did not produce a DataFrame and capture metadata.")


Python: 3.13.5
Kernel: /Users/onurerbey/Library/Mobile Documents/com~apple~CloudDocs/GitHub/nyc-311-data-vault-dbt/.venv/bin/python
Ready to request up to 1,000 rows; no request sent yet.
Loaded 1,000 rows and 44 source columns into df.
Sample limit reached; additional matching records may exist.


## Run the table creation SQL

Register the response as `api_service_requests` in the same DuckDB connection that executes the SQL file. Persist `snapshot_at_utc` and `record_source` alongside the selected source columns; source date strings retain their original meaning. Rerunning this cell replaces the sample table using the currently loaded `df`.

In [13]:
if not isinstance(df, pd.DataFrame) or capture_metadata is None:
    raise RuntimeError("Run the API fetch cell successfully before creating the table.")

api_service_requests = df.assign(
    snapshot_at_utc=pd.Timestamp(capture_metadata["snapshot_at_utc"]),
    record_source=capture_metadata["record_source"],
)
sql_query = SQL_PATH.read_text(encoding="utf-8")
DATABASE_PATH.parent.mkdir(parents=True, exist_ok=True)
with duckdb.connect(str(DATABASE_PATH)) as con:
    con.register("api_service_requests", api_service_requests)
    con.execute(sql_query)
    row_count = con.sql('SELECT count(*) FROM "311_service_requests"').fetchone()[0]
print(f"Created 311_service_requests with {row_count:,} rows in {DATABASE_PATH}.")


Created 311_service_requests with 1,000 rows in /Users/onurerbey/Library/Mobile Documents/com~apple~CloudDocs/GitHub/nyc-311-data-vault-dbt/data/nyc_open_data.db.


## Query the saved table

This step also works in a fresh kernel after running just the setup cell. The saved table remains available after the API DataFrame and registration are gone.

In [14]:
sql_query = '''
SELECT unique_key, created_date, agency, complaint_type, status, borough
FROM "311_service_requests"
ORDER BY created_date DESC NULLS LAST, unique_key DESC
LIMIT 10
'''
with duckdb.connect(str(DATABASE_PATH), read_only=True) as con:
    display(con.sql(sql_query).df())


,unique_key,created_date,agency,complaint_type,status,borough
0,70774279,2026-10-17T02:06:37.000,NYPD,Noise - Street/Sidewalk,Unspecified,Unspecified
1,70783143,2026-10-17T02:06:14.000,TLC,Taxi Complaint,In Progress,QUEENS
2,70779590,2026-10-17T02:06:13.000,NYPD,Noise - Residential,In Progress,Unspecified
3,70785024,2026-10-17T02:05:51.000,NYPD,Noise - Commercial,In Progress,QUEENS
4,70777529,2026-10-17T02:05:41.000,NYPD,Noise - Residential,In Progress,BROOKLYN
5,70783555,2026-10-17T02:05:37.000,NYPD,Noise - Residential,In Progress,BRONX
6,70775768,2026-10-17T02:05:03.000,DEP,Noise,In Progress,BROOKLYN
7,70780367,2026-10-17T02:04:49.000,NYPD,Illegal Parking,In Progress,BROOKLYN
8,70775208,2026-10-17T02:04:44.000,NYPD,Noise - Street/Sidewalk,In Progress,BRONX
9,70779417,2026-10-17T02:04:36.000,NYPD,Noise - Street/Sidewalk,In Progress,BRONX
